In [1407]:
import numpy as np
import scipy.optimize as opt
import serial
import serial.tools.list_ports as ports
import time
import can
import struct


# ----- RS485 COMMAND SET -----
def moveTo_XY(x, y):
    """
        Commands X-Y stage to move to specified point in x-y plane
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    if np.allclose(x, readPosition(3)) and np.allclose(y, readPosition(4)):
        return
         
    #Calculating steps based on distance and physical properties of rails
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    #match move times   
    if (np.abs(x-readPosition(3))*(np.abs(y-readPosition(4))) != 0):
        speed_X = (np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))])))
        speed_Y = (np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))])))
        acc_X = (np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = (np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, int(speed_X), int(acc_X))
        setSpeedAcc(4, int(speed_Y), int(acc_Y))
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
    
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"
    
    #Command Motors
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.05)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.05)
    ser.write(runString.encode('ascii'))
    time.sleep(0.05)
    print(f"XY command sent. Moving to ({x}, {y})")


def moveRelative_XY(x, y):
    """
        Commands X-Y stage to move to specified point in x-y plane relative to its current position
        Inputs:
            x: float: X distance to move to in mm 
            y: float: Y distance to move to in mm 
    """
    if x == 0 and y == 0:
        return
    #Calculating steps based on distance and physical properties of rails
    steps_X = int(abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(abs(y*stepsPerRotation_XY/mmPerThread))

    #match move times
    if x*y !=0:
        speed_X = int(np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x)/np.abs(y))])))
        speed_Y = int(np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y)/np.abs(x))])))
        acc_X = int(np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = int(np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, speed_X, acc_X)
        setSpeedAcc(4, speed_Y, acc_Y)
    elif (x == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (y == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        
    #Create String to send
    if x>0:
        moveString_X = f"/3P{str(steps_X)}<CR>\r\n"
    elif x<0:
        moveString_X = f"/3D{str(steps_X)}<CR>\r\n"
    else:
        moveString_X = f"/3P1<CR>\r\n" #Find a way to fix this ideally. when the realtive position is set to zero weird things happen, so i just make it 1 artifiacially so it barely moves
    if y>0:
        moveString_Y = f"/4P{str(steps_Y)}<CR>\r\n"
    elif y<0:
        moveString_Y = f"/4D{str(steps_Y)}<CR>\r\n"
    else:
        moveString_Y = f"/4P1<CR>\r\n"
    runString = f"/CR\r\n"

    #Command Motors
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    print(f"XY command sent. Moving by ({x}, {y}) to ({x + readPosition(3)}, {y + readPosition(4)})")
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)

def moveTo_Ze(theta):
    """
        Commands zenith motor to move to specified point 
        Inputs:
            theta: float: location to move to (degrees)
    """
    steps_Ze = int(200*16*theta/360) #steps*microsteps*angle in degrees
    
    moveTo_CAN(1, steps_ze)
    time.sleep(0.05)
    setSpeedAcc(1, maxVel_Ze, maxAcc_Ze)
    time.sleep(0.05)
    begin_CAN(1)
    
    print(f"Ze command sent. Moving zenith axis to {round(theta, 3)} degrees")


def moveRelative_Ze(theta):
    """
        Commands Zenith motor to move a certain amount from its current position
        Inputs:
            theta: float: distance to move in degrees
    """
    if theta == 0:
        return
    
    steps_Ze = int(200*16*theta/360) #steps*microsteps*angle in degrees
    
    moveRelative_CAN(1, steps_ze)
    time.sleep(0.05)
    setSpeedAcc(1, maxVel_Ze, maxAcc_Ze)
    time.sleep(0.05)
    begin_CAN(1)
    print(f"Ze command sent. Moving zenith axis by {round(theta, 3)} degrees")

def moveTo_Az(theta):
    """
        Commands azimuth motor to move to specified point 
        Inputs:
            theta: float: location to move to (degrees)
    """
    steps_Az = int(6*200*16*theta/360) #gear ratio*steps*microsteps*angle in degrees

    moveTo_CAN(2, steps_Az)
    time.sleep(0.05)
    setSpeedAcc(2, maxVel_Az, maxAcc_Az)
    time.sleep(0.05)
    begin_CAN(2)
    
    print(f"Ze command sent. Moving azimuth axis to {round(theta, 3)} degrees")


def moveRelative_Az(theta):
    """
        Commands azimuth motor to move a certain amount from its current position
        Inputs:
            theta: float: distance to move in degrees
    """
    if theta == 0:
        return
    
    steps_Ze = int(6*200*16*theta/360) #gear ratio*steps*microsteps*angle in degrees
    
    moveRelative_CAN(2, steps_Az)
    time.sleep(0.05)
    setSpeedAcc(2, maxVel_Az, maxAcc_Az)
    time.sleep(0.05)
    begin_CAN(2)
    print(f"Ze command sent. Moving azimuth axis by {round(theta, 3)} degrees")


def moveToAll_XY_smooth(x, y):
    """
        Moves all motors so that the the heat source is pointed towards a target some distance (H defined below) above (x, y) = (275, 275)
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    x, y = findXCYC(x, y)
    #Calculating steps based on distance and physical properties of rails---------------------------------------------
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    # matching Move times---------------------------------------------------------------------------------------------
    if not np.allclose (np.abs(x-readPosition(3))*(np.abs(y-readPosition(4))), 0):
        speed_X = (np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))])))
        speed_Y = (np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))])))
        acc_X = (np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = (np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, int(speed_X), int(acc_X))
        setSpeedAcc(4, int(speed_Y), int(acc_Y))
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        
    # #Create String to send-----------------------------------------------------------------------------------------
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"


    #Command Motors--------------------------------------------------------------------------------------------------
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)

    while (not np.allclose(x,readPosition(3))) or (not np.allclose(y, readPosition(4))):
        moveTo_Ze(findTheta(readPosition(3), readPosition(4))*180/np.pi)
        if readPosition(4) != 275:
            moveTo_Az(int(-(np.arctan2((readPosition(3)-275), (readPosition(4)-275))*180/(np.pi)-(2/8)*360)))
        else:
            moveTo_Az(int(-((np.pi/2)*180/(np.pi)-(2/8)*360)))

    moveTo_Ze(findTheta(x, y)*180/np.pi)
    if readPosition(4) != 275:
        moveTo_Az(int(-(np.arctan2((readPosition(3)-275), (readPosition(4)-275))*180/(np.pi)-(2/8)*360)))
    else:
        moveTo_Az(int(-((np.pi/2)*180/(np.pi)-(2/8)*360)))
    
def moveToAll_XY(x, y):
    """
        Moves all motors so that the the heat source is pointed towards a target some distance (H defined below) above (x, y) = (275, 275)
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    x, y = findXCYC(x, y)
    #Calculating steps based on distance and physical properties of rails---------------------------------------------
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    # matching Move times---------------------------------------------------------------------------------------------
    if not np.allclose (np.abs(x-readPosition(3))*(np.abs(y-readPosition(4))), 0):
        speed_X = (np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))])))
        speed_Y = (np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))])))
        acc_X = (np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = (np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, int(speed_X), int(acc_X))
        setSpeedAcc(4, int(speed_Y), int(acc_Y))
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        
    # #Create String to send-----------------------------------------------------------------------------------------
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"


    #Command Motors--------------------------------------------------------------------------------------------------
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)

    setSpeedAcc(2, int(maxVel_Az), int(maxAcc_Az))
    setSpeedAcc(1, int(maxVel_Ze), int(maxAcc_Ze))

    moveTo_Ze(findTheta(x, y)*180/np.pi)
    time.sleep(0.1)
    moveTo_Az(findPhi(x, y)*6*3200/(2*np.pi))
    time.sleep(0.1)

def moveToAll_thetaPhi(theta, phi):
    """
        Moves all motors so that the the heat source is pointed towards a target some distance (H defined below) above (x, y) = (275, 275)
        Inputs:
            x: float: X location to move to in mm (0-550)
            y: float: Y location to move to in mm (0-550)
    """
    x, y = findXY(theta, phi)
    if (np.abs(x - d/2) > d/2) or (np.abs(y - d/2) > d/2):
        print(f"Out of bounds request error. ({x}, {y})")
        return
    #Calculating steps based on distance and physical properties of rails---------------------------------------------
    steps_X = int(np.abs(x*stepsPerRotation_XY/mmPerThread))
    steps_Y = int(np.abs(y*stepsPerRotation_XY/mmPerThread))

    # matching Move times---------------------------------------------------------------------------------------------
    if (not np.allclose(np.abs(x-readPosition(3))*(np.abs(y-readPosition(4))), 0)):
        speed_X = (np.min(np.array([maxVel_X, maxVel_Y*(np.abs(x-readPosition(3))/np.abs(y-readPosition(4)))])))
        speed_Y = (np.min(np.array([maxVel_Y, maxVel_X*(np.abs(y-readPosition(4))/np.abs(x-readPosition(3)))])))
        acc_X = (np.min(np.array([maxAcc_X, maxAcc_Y*speed_X/speed_Y])))
        acc_Y = (np.min(np.array([maxAcc_Y, maxAcc_X*speed_Y/speed_X])))
        setSpeedAcc(3, int(speed_X), int(acc_X))
        setSpeedAcc(4, int(speed_Y), int(acc_Y))
    elif (np.abs(x-readPosition(3)) == 0):
        setSpeedAcc(4, maxVel_Y, maxAcc_Y)
    elif (np.abs(y-readPosition(4)) == 0):
        setSpeedAcc(3, maxVel_X, maxAcc_X)
        
    # #Create String to send-----------------------------------------------------------------------------------------
    moveString_X = f"/3A{str(steps_X)}<CR>\r\n"
    moveString_Y = f"/4A{str(steps_Y)}<CR>\r\n"
    runString = f"/CR\r\n"


    #Command Motors--------------------------------------------------------------------------------------------------
    ser.write(moveString_X.encode('ascii'))
    time.sleep(0.1)
    ser.write(moveString_Y.encode('ascii'))
    time.sleep(0.1)
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)
    
    moveTo_Ze(theta)
    time.sleep(0.1)
    moveTo_Az(phi)

def zero(motor):
    """
        Sets specified motor(s) be at zero at its current position
        Inputs:
          "_": all
          "C": Motors 3 and 4 (x-y stage)
            1: ze
            2: az
            3: X
            4: Y
            5: chop
    """
    if motor == 1 or motor == 2:
        zero_CAN(motor)
        return
        
    reset = f"/{motor}z0R\r\n"
    ser.write(reset.encode('ascii'))
    print(f"Motor {motor} has been zeroed")



def highZero(motor):
    """
        Sets specified motor(s) to be at a large number position for purposes of setting the zero value
        Inputs:
          "_": all
          "C": Motors 3 and 4 (x-y stage)
            1: ze
            2: az
            3: X
            4: Y
            5: chop
    """
    reset = f"/{motor}z100000000R\r\n"
    ser.write(reset.encode('ascii'))
    print(f"Motor {motor} has been set high")
    
def startChop(Hz):
    """
        turns on the chopper motor to turn indefinetly at a specified frequency
        Inputs: Hz: float: the frequency that the chopper will chop the hear source at (Hz)

        negetive input spins CW, positive spins CCW    
    """
    #set up chop
    initString_Chop = f"/5m{curr_Chop}V{str(np.abs(Hz*200*256//2))}L{maxAcc_Chop}R\r\n" #for velocity: Hz * full steps * microsteps * 1/2 (chopper wheel has 2 on/offs per rotation)
    ser.write(initString_Chop.encode("ascii"))
    time.sleep(0.5)
    if Hz > 0:
        go = f"/5P0<CR>\r\n"
    elif Hz < 0:
        go = f"/5D0<CR>\r\n"
    else:
        return
    runString = f"/5R\r\n"
    
    #Command Motors
    ser.write(go.encode('ascii'))
    time.sleep(0.5)
    ser.write(runString.encode('ascii'))
    time.sleep(0.5)
    print("Chop started")
    
def stopChop():
    """
        Stops chopper motor
    """
    runString = f"/5TR\r\n"
    ser.write(runString.encode('ascii'))
    print("Chop stopped")

def setSpeedAcc(motor, speed, acc):
    """
        Sets the speed and acceleration for any given motor in the system
        Inputs:
            Motor: 
                "_": all
                "C": Motors 3 and 4 (x-y stage)
                1: ze
                2: az
                3: X
                4: Y
                5: chop
            Speed: float: value to set motor speed to in steps/sec
            Acc: float: value to set motor acceleration to in steps/sec^2
    """
    if motor == 1 or motor == 2:
        setSpeed_CAN(motor, speed)
        time.sleep(0.1)
        setAcc_CAN(motor, acc)
        time.sleep(0.1)
        return
        
    initString = f"/{int(motor)}V{speed}L{acc}R\r\n"
    ser.write(initString.encode("ascii"))
    time.sleep(0.1)    

def readPosition(motor, timeout = 0.1):
    """
       gets the current position of any motor
       Inputs:
           motor: int: Motor for which the position is read
               1: ze
               2: az
               3: X
               4: Y
               5: chop
            timeout: float: time to wait in seconds before giving up on position read
        Outputs:
            Position: int: 
               1: ze in degrees
               2: az in degrees
               3: X in mm
               4: Y in mm
               5: chop in number of rotations
                
    """
    if motor == 1 or motor == 2:
        readPosition_CAN(motor)
        return None
        
    # Flush any leftover data in the buffers
    ser.reset_input_buffer()
    ser.reset_output_buffer()

    command = f"/{motor}?0R\r\n"
    ser.write(command.encode('ascii'))

    time.sleep(0.05)
    response = ser.readline().decode('ascii', errors='ignore').strip()
    
    if motor == 3 or motor == 4:     
        return int(response[3:-1])/(stepsPerRotation_XY/mmPerThread)
    elif motor == 5:
        return int(response[3:-1])/(200*256)
    elif motor ==1:
        return int(response[3:-1])/(200*256)*360
    

def endAll():
    """
        Stops all motors
    """
    stopMotion_CAN(1)
    time.sleep(0.1)
    stopMotion_CAN(2)
    time.sleep(0.1)
    runString = f"/_TR\r\n"
    ser.write(runString.encode('ascii'))
    time.sleep(0.1)
    stopMotion_Az()
    time.sleep(0.1)
    print("Current commands halted")

def close():
    """
        Closes serial port when done with beam mapper
    """
    ser.close()
    print(f"Port Closed")


# ----- CAN COMMAND SET -----
CAN_MO = 0x00000095   # Driver State (on/off)
CAN_SP = 0x0000009E   # PTP motor speed
CAN_PA = 0x000000A0   # Absolute position
CAN_BG = 0x00000096   # Begin Motion
CAN_ZP = 0x000000A1   # Set position (zero)
CAN_IC = 0x00000086   # Internal Configuration
CAN_PA = 0x000000A0   # Absolute Position
CAN_AC = 0x00000090   # Acceleration
CAN_DC = 0x00000091   # Deceleration
CAN_ST = 0x00000097   # Stop motion
CAN_JV = 0x0000009D   # Jog velocity
CAN_SY = 0x0000007E   # System Operation

def motorOn_CAN(motor):
    """
        turns specified motor driver on
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """        
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_MO)),
        data=[0x01],
        is_extended_id=True
    ))
    time.sleep(0.05)

def motorOff_CAN(motor):
    """
        turns specified motor off
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_MO)),
        data=[0x00],
        is_extended_id=True
    ))

def enableClosedLoop_CAN(motor):
    """
        enables closed loop control of specified motor
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_IC)),
        data=[0x06, 0x01, 0x00],
        is_extended_id=True
    ))
    time.sleep(0.1)

def setAcc_CAN(motor, accel):
    """
        sets the acceleration of specified motor
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
            accel: int: acceleration of motor (pulse/sec^2)
    """        
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_AC)),
        data=struct.pack("<i", accel),
        is_extended_id=True
    ))

def setDecel_CAN(motor, decel):
    """
        sets the deceleration of specified motor
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
            decel: int: decceleration of motor (pulse/sec^2)
    """        
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_DC)),
        data=struct.pack("<i", decel),
        is_extended_id=True
    ))
    time.sleep(0.05)

def zero_CAN(motor):
    """
        sets the current specified motor position to be the zero position
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_ZP)),
        data=struct.pack("<i", 0),
        is_extended_id=True
    ))

def moveTo_CAN(motor, position):
    """
        commanads specified motor to move to a specific position
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
            position: int: position to move to (steps) (3200 in one rotation)
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_PA)),
        data=struct.pack("<i", int(position)),
        is_extended_id=True
    ))
    print(f"Az command sent. Moving azimuth axis to {round(position/(6*16*200)*360, 3)} degrees")
    
def moveRelative_CAN(motor, position):
    """
        commanads specified motor to move a certain amount
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
            position: int: amount to move by (steps) (3200 in one rotation)
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_PR)),
        data=struct.pack("<i", int(position)),
        is_extended_id=True
    ))
    
def setSpeed_CAN(motor, speed):
    """
        sets the speed of specified motor
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
            speed: int: speed of motor (pulse/sec)
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_SP)),
        data=struct.pack("<i", int(speed)),
        is_extended_id=True
    ))
    
def begin_CAN(motor):
    """
        activates newly set parameters and initiates motion. must be ran for motor to move
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_BG)),
        data=[],
        is_extended_id=True
    ))


def stopMotion_CAN(motor):
    """
        stops current motor movement
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_ST)),
        data=[],
        is_extended_id=True
    ))

def clearJogMode_CAN(motor):
    """
        sets jog velocity to zero. Used to enter PTP mode if stuck
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    # JV = 0 → exit jog / velocity mode
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_JV)),
        data=struct.pack("<i", 0),
        is_extended_id=True
    ))


def readPosition_CAN(motor, timeout=1.0):
    """
        Requests the current absolute encoder position of specified motor from the UIM342.
        Returns position (int) or None.
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_PA)),
        data=[],
        is_extended_id=True
    ))

    t0 = time.time()
    while time.time() - t0 < timeout:
        msg = bus.recv(timeout=0.1)
        if msg is None:
            continue

        cw = msg.arbitration_id & 0xFF

        if cw == 0x20 and msg.dlc >= 4:
            pos = struct.unpack("<i", msg.data[:4])[0]
            return pos

    return None

def invertEncoderDirection_CAN(motor):
    """
        Inverts the encoder direction of specified motor. USE WITH CAUTION!!!
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """
    # IC[5] = 1 → invert encoder polarity
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_IC)),
        data=[0x05, 0x01, 0x00],
        is_extended_id=True
    ))

def restoreFactoryDefaults_CAN(motor):
    """
        sets specified motor back to factory settings
        Inputs:
            motor: int: number of motor to command
                1: ze
                2: az
    """        
    bus.send(can.Message(
        arbitration_id=hex(int(0x04000000 + motor*0x00080000+CAN_SY)),
        data=[0x02],
        is_extended_id=True
    ))

def changeMotorID():
    """
        Changes the ID number that the currently connetce dmotor is addressed with
        Input:
            ID: int: new ID of the motor (change manually in last entry of data array)
    """
    bus.send(can.Message(
        arbitration_id=0x04000081,
        data=[0x07, 0x01],
        is_extended_id=True
     ))
# -----------------------------    
    

In [1365]:
port_CAN = 'COM5' # Change according to your computer. Look in Device Manager
port_RS485 = 'COM3' # Change according to your computer. Look in Device Manager

baud = 9600 # for RS485
BITRATE = 500000 #for CAN

#Initialization for RS485 Port
ser = serial.Serial(port_RS485, baudrate = baud, timeout = 1)
time.sleep(0.5)
ser.reset_input_buffer()
ser.reset_output_buffer()

#initiates CAN bus port
bus = can.interface.Bus(
    interface="slcan",
    channel=port_CAN,
    bitrate=BITRATE
    )
motorOn_CAN(1)
time.sleep(0.05)
motorOn_CAN(2)
print("Ports active")

Ports active


In [1366]:
#Motor Numbers:
#   Zenith: 1 (6*200*16 for one full rotation of az-ze stage)
#   Azimuth: 2
#   X: 3
#   Y: 4
#   Chop: 5

stepsPerRotation_XY = 200*256 #full steps * number of microsteps
mmPerThread = 5 # 5 was roughly measured (dont ask Owen about this)


maxVel_X = 90000 #Microsteps/Second (90000 MAX works for 256 microsteps) (int)
maxAcc_X = 100 #Microsteps/Second^2 / (400,000,000/65536) (int)
curr_X = 100 #Movement Current as a percentage of max (Typically Use 100) (int)

maxVel_Y = 200000 #Microsteps/Second (400000 MAX works for 256 microsteps) (int)
maxAcc_Y = 100 #Microsteps/Second^2 / (400,000,000/65536) (int)
curr_Y = 100 #Movement Current as a percentage of max (Typically Use 100) (int)

maxVel_Ze = 50000 #Microsteps/Second (50000 works for 256 microsteps) (int)
maxAcc_Ze = 100 #Microsteps/Second^2 / (400,000,000/65536)  (int)
curr_Ze = 100 #Movement Current as a percentage of max (Typically Use 100) (int)

maxVel_Az = 32000 #Microsteps/Second (32000 works fast and safe for 16 microsteps) (int)
maxAcc_Az = 100 #Microsteps/Second^2  (int) FIGURE THIS OUT
# curr_Az = 2 #current working current (not sure how this effects movement so leaving it at the deffault for now) (int)

curr_Chop = 100 #Movement Current as a percentage of max (Typically Use 100) (int)
maxAcc_Chop = 100 #Microsteps/Second^2 / (400,000,000/65536) (int)

# k = 10 #heat source height above zenith rotation axis (mm) (dont think we actually need)
g = -35 #distance of heat source from center of azimuth rotation axis in direction parllel to zenith rotation axis (pos. towards az Motor)(mm)
h = 1255 #vertical distnace from center of zenith rotation axis to point where the heat source should be aimed at (mm)
j = 0 #distance of heat source from center of azimuth rotation axis in direction perpendicular to zenith rotation axis (pos. opposite dir. of chop motor set screw) (mm)

d = 550 #max x/y distance to travel. used so rail limit is not exceeded (mm)

In [1367]:

initString_X = f"/3m{curr_X}V{maxVel_X}L{maxAcc_X}R\r\n"
initString_Y = f"/4m{curr_Y}V{maxVel_Y}L{maxAcc_Y}R\r\n"

#Initialization for RS485 Port
ser.write(initString_X.encode("ascii"))
time.sleep(0.1)
ser.write(initString_Y.encode("ascii"))
time.sleep(0.1)

setSpeedAcc(1, maxVel_Ze, maxAcc_Ze)
time.sleep(0.05)
setSpeedAcc(2, maxVel_Az, maxAcc_Az)
time.sleep(0.05)
setSpeedAcc(3, maxVel_X, maxAcc_X)
time.sleep(0.05)
setSpeedAcc(4, maxVel_Y, maxAcc_Y)
time.sleep(0.05)
setSpeedAcc(5, maxVel_Chop, maxAcc_Chop)
time.sleep(0.05)

print("Initiation Complete")

Initiation Complete


In [1396]:
#math Functions

def findTheta(X_A, Y_A, G = g, H = h):
    """
        determines the amount to angle the heat source given its postion
        Inputs:
            X_A: float: X position (mm)
            X_A: float: Y position (mm)
        Outputs:
            theta: float: Angle of heat source in radians
    """
        
    def thetaFunc(theta, X_A, Y_A, H):
        return np.arctan(((((X_A-275)**2)+((Y_A-275)**2))**0.5)/(H-j*np.sin(theta)))-theta

        
    return opt.brentq(thetaFunc, -np.pi, np.pi/2, args = (X_A, Y_A, H))

def findPhi(X_A, Y_A, G = g, H = h):
    """
        determines the amount rotate the az stage given xy position
        Inputs:
            X_A: float: X position (mm)
            X_A: float: Y position (mm)
        Outputs:
            phi: float: Angle of az stage in radians
    """
    
    phi = -(np.arctan2((X_A-275), (Y_A-275)) - np.pi/2)
    return phi

def findXCYC(x, y):
    return np.array([x+g*np.sin(findPhi(x, y))-j*np.cos(findPhi(x, y)), y-g*np.cos(findPhi(x, y))-j*np.sin(finPhi,(x, y))])

def findXY(theta, phi):
    x = -h*np.tan(theta)*np.cos(phi)+(j/np.cos(theta))*np.cos(phi)-g*np.sin(phi)+d/2
    y = -h*np.tan(theta)*np.sin(phi)+(j/np.cos(theta))*np.sin(phi)+g*np.cos(phi)+d/2 
    return np.array([x, y])
                     


In [1397]:
def moveToPoints_XY(points):
    """
        commands XY stage to move to a set of speciffied points in the XY plane one after another
        Inputs:
            points: nx2 array of x-y points to move to in mm
    """
    numPoints = points.shape[0]

    for point in range(numPoints):
        moveTo_XY(points[point][0], points[point][1])
        while not (np.allclose(points[point][0], readPosition(3)) or np.allclose(points[point][1], readPosition(4))):
            time.sleep(0.01)

def moveToPoints_thetaPhi(points):
    """
        commands XY and az-ze stage to move to a set of speciffied points one after another, while keeping pointing at the same spot
        Inputs:
            points: nx2 array of theta and Phi coordinate points to move to in mm
    """
    numPoints = points.shape[0]

    for point in range(numPoints):
        moveToAll_thetaPhi(points[point][0], points[point][1])
        while not (np.allclose(points[point][0], readPosition(3)) or np.allclose(points[point][1], readPosition(4))):
            time.sleep(0.01)
def makeThetaPhiPoints(theta, phi, thetaPoints, phiPoints, degrees = True):
    """
        creates a set of points for the beam mapper to map
        Inputs:
            theta: touple: min and max theta ranges
            phi: touple: min and max phi ranges
            thetaPoints: int: number of different theta points to travel to
            phiPoint:int: number of differrent phi points to travel to
            degrees: Bool: if True makes inputs in units of degrees. Else makes units of radians
        Outputs: array: thetaPoints*phiPoints x 2 array of all coordinate points to move to
    """
    thetaArray = np.array(np.linspace(theta[0], theta[1], thetaPoints))
    phiArray = np.array(np.linspace(phi[0], phi[1], phiPoints))

    A, B = np.meshgrid(thetaArray, phiArray)

    points = np.vstack([A.ravel(), B.ravel()]).T

    if degrees:
        return points
    else:
        return points*np.pi/180
    


In [1379]:
vals = np.linspace(0, 2*np.pi, 50)

vals_X = 100*np.cos(vals)+275
vals_Y = 100*np.sin(vals)+275

matrix = np.column_stack((vals_X, vals_Y))


In [1376]:
close()
bus.shutdown()

Port Closed


In [1207]:
highZero(4)

Motor 4 has been set high


In [1329]:
zero(1)

Motor 1 has been zeroed


In [1330]:
moveTo_XY(0, 0)

In [1212]:
moveRelative_XY(0, -10)

XY command sent. Moving by (0, -10) to (9525.6251953125, 9495.625)


In [1337]:
print(readPosition(1))

12.965625000000001


In [1334]:
moveRelative_Ze(10)

Ze command sent. Moving zenith axis by 10 degrees


In [1282]:
moveTo_Ze(0)

Ze command sent. Moving zenith axis to 0 degrees


In [1360]:
string = f"/1P1000R\r\n"
ser.write(string.encode('ascii'))

10

In [1279]:
moveTo_all(275, 275)

Out of bounds request error. (1487.540097622066, -10956.273443746702)


In [ ]:
endAll()

In [759]:
startChop(100)

/5P0<CR>

Chop started


In [760]:
stopChop()

In [1213]:
zero("C")

Motor C has been zeroed


In [ ]:
motorOn_Az()
setSpeedAcc(2, 0.1*maxVel_Az, maxAcc_Az)
moveTo_Az(0*6*200*16)
begin_Az()

In [1118]:
zero_Az()

In [1238]:

moveToAll_thetaPhi(0*np.pi/15, -np.pi/4)

Ze command sent. Moving zenith axis to 0.0 degrees
Az command sent. Moving azimuth axis to -45.0 degrees


In [1398]:
print(hex(int(0x04000000 + 3*0x00080000+0x00000095)))

0x4180095


In [1400]:
motor = 2

print(hex(int(0x04000000 + motor*0x00080000+CAN_MO)))

0x4100095
